# Diabetes classification with accuracy as the primary endpoint

Tune the classifier and its decision cutoff together under a fixed search budget. Compare the procedure with majority-class prediction and default XGBoost on the same nested cross-validation folds. The earlier notebook's 154 test rows are excluded from this experiment.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, make_scorer
from sklearn.model_selection import GridSearchCV, ParameterSampler, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

data_path = Path("data/diabetes.csv")
if not data_path.is_file():
    raise FileNotFoundError(f"Place the project CSV at {data_path} before running the notebook.")

data = pd.read_csv(data_path)
features = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age",
]
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
X = data[features].copy()
X[zero_as_missing] = X[zero_as_missing].replace(0, np.nan)
y = data["Outcome"]
if y.isna().any() or set(y.unique()) != {0, 1}:
    raise ValueError("Outcome must contain both binary labels (0 and 1) and no missing values.")

seed = 42
X_dev, _, y_dev, _ = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=seed
)

## Search budget and objective

Sample 18 XGBoost configurations from a broad space and test six regularized logistic regression configurations. Each candidate fits once per inner fold. Score seven cutoffs from 0.35 to 0.65 using the same validation probabilities, so threshold search requires no additional model fits. Select the candidate and cutoff with the highest mean validation accuracy. When cutoff scores tie, prefer the one closest to 0.50.

In [ ]:
xgb = XGBClassifier(random_state=seed, n_jobs=1, tree_method="hist")
xgb_space = {
    "model__max_depth": [2, 3, 4, 5],
    "model__learning_rate": [0.025, 0.05, 0.1, 0.2],
    "model__n_estimators": [80, 150, 250, 400],
    "model__min_child_weight": [1, 3, 6],
    "model__subsample": [0.75, 1.0],
    "model__colsample_bytree": [0.75, 1.0],
    "model__reg_lambda": [0.5, 1.0, 3.0, 10.0],
}
sampled_xgb = list(ParameterSampler(xgb_space, n_iter=18, random_state=seed))
candidates = [
    {"model": [xgb], **{key: [value] for key, value in params.items()}}
    for params in sampled_xgb
]
candidates.append({
    "scale": [StandardScaler()],
    "model": [LogisticRegression(max_iter=1000)],
    "model__C": [0.1, 1.0, 10.0],
    "model__class_weight": [None, "balanced"],
})

pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", "passthrough"),
    ("model", xgb),
])

def cutoff_accuracy(y_true, scores, cutoff):
    return accuracy_score(y_true, scores >= cutoff)

cutoffs = [0.50, 0.45, 0.55, 0.40, 0.60, 0.35, 0.65]
scorers = {
    f"cutoff_{index}": make_scorer(
        cutoff_accuracy, response_method="predict_proba", cutoff=cutoff
    )
    for index, cutoff in enumerate(cutoffs)
}

def score_matrix(cv_results):
    return np.vstack([cv_results[f"mean_test_{name}"] for name in scorers])

def best_candidate(cv_results):
    scores = score_matrix(cv_results)
    return int(np.unravel_index(scores.argmax(), scores.shape)[1])

def chosen_cutoff(fitted_search):
    scores = score_matrix(fitted_search.cv_results_)
    return cutoffs[scores[:, fitted_search.best_index_].argmax()]

inner_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=seed)
search = GridSearchCV(
    pipeline, candidates, scoring=scorers, refit=best_candidate,
    cv=inner_cv, error_score="raise",
)

## Estimate accuracy with nested cross-validation

The outer evaluation predicts each development row once using a model and cutoff selected without that row. Compare pooled accuracy across the same rows and report sensitivity and specificity to show what the accuracy choice costs. The 24 candidates require 96 model fits per search before refitting the winner.

In [ ]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed + 1)
default_xgb = make_pipeline(
    SimpleImputer(strategy="median"),
    XGBClassifier(random_state=seed, n_jobs=1),
)
predictions = {
    label: np.empty(len(y_dev), dtype=int)
    for label in ("Majority class", "Default XGBoost", "Selected model, cutoff 0.50", "Joint accuracy search")
}
fold_choices = []
for fold, (fit_rows, score_rows) in enumerate(outer_cv.split(X_dev, y_dev), start=1):
    X_fit, y_fit = X_dev.iloc[fit_rows], y_dev.iloc[fit_rows]
    X_score = X_dev.iloc[score_rows]
    fitted_search = clone(search).fit(X_fit, y_fit)
    cutoff = chosen_cutoff(fitted_search)
    scores = fitted_search.predict_proba(X_score)[:, 1]

    majority = DummyClassifier(strategy="most_frequent").fit(X_fit, y_fit)
    predictions["Majority class"][score_rows] = majority.predict(X_score)
    predictions["Default XGBoost"][score_rows] = clone(default_xgb).fit(X_fit, y_fit).predict(X_score)
    predictions["Selected model, cutoff 0.50"][score_rows] = scores >= 0.50
    predictions["Joint accuracy search"][score_rows] = scores >= cutoff
    model_name = type(fitted_search.best_estimator_.named_steps["model"]).__name__
    fold_choices.append({"Fold": fold, "Model": model_name, "Cutoff": cutoff})

def summarize(y_true, predicted):
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    return {
        "Accuracy": accuracy_score(y_true, predicted),
        "Sensitivity": tp / (tp + fn),
        "Specificity": tn / (tn + fp),
        "False positives": fp,
        "False negatives": fn,
    }

print(pd.DataFrame(fold_choices).to_string(index=False))
results = pd.DataFrame.from_dict(
    {name: summarize(y_dev, pred) for name, pred in predictions.items()}, orient="index"
)
results.round(4)

## Refit on all development rows

This produces a candidate to carry forward. Its parameters and cutoff can differ from the fold-specific choices. The nested comparison estimates the performance of the whole selection procedure. Evaluate the final fitted model on fresh data before treating the accuracy as independently confirmed.

In [ ]:
final_search = clone(search).fit(X_dev, y_dev)
print("Model:", type(final_search.best_estimator_.named_steps["model"]).__name__)
print("Hyperparameters:", {
    key.removeprefix("model__"): value
    for key, value in final_search.best_params_.items()
    if key.startswith("model__")
})
print(f"Cutoff: {chosen_cutoff(final_search):.2f}")

## Reflection

This generation makes the optimization objective and search cost explicit. On the development folds, joint tuning gained three correct predictions over untuned XGBoost. Keeping the same selected models and applying a 0.50 cutoff gained four correct predictions over joint tuning. The cutoff comparison shows how the error tradeoff changes under an accuracy objective. The earlier test rows were excluded, so compare these figures within this experiment.